# 63: Conditional GAN for Noisy Drone RF v2 Signatures

This notebook uses the successful `33_vgg_spectrogram_noisy_drone_rf_v2.ipynb` pipeline as a teacher-guided generator experiment.

Goal: train a conditional generator that can synthesize compact spectrogram “signatures” for each drone/controller class (`DJI`, `FutabaT14`, etc.). The GAN learns from real Noisy Drone RF v2 I/Q captures and optionally uses the frozen canonical `33` VGG model as a class-consistency teacher.

This does **not** replace the classifier. It is a research notebook for data augmentation, signature visualization, and eventually synthetic pretraining.

Update: this notebook defaults to frozen-discriminator continuation mode (`NOISY_DRONE_GAN_FREEZE_DISCRIMINATOR=1`) because the discriminator can classify generated signatures while the frozen `33` VGG teacher may still collapse them to `Noise`.


In [ ]:
# Cell 1: Configure paths, labels, and GAN training knobs
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import sys
if 'ipykernel' not in sys.modules and not os.environ.get('DISPLAY'):
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import torch
from sklearn.model_selection import train_test_split
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    BatchNormalization,
    Concatenate,
    Conv2D,
    Conv2DTranspose,
    Dense,
    Dropout,
    Embedding,
    Flatten,
    Input,
    LeakyReLU,
    Reshape,
)
from tensorflow.keras.models import load_model

try:
    project_root = Path(__file__).resolve().parents[1]
except NameError:
    project_root = Path.cwd().resolve()
    if project_root.name == 'notebooks':
        project_root = project_root.parent

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
tf.get_logger().setLevel('ERROR')

outputs_dir = project_root / 'outputs' / 'noisy_drone_rf_v2_gan'
model_dir = project_root / 'models' / 'noisy_drone_rf_v2'
outputs_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

teacher_model_path = model_dir / 'noisy_drone_rf_v2_vgg_full_complex_spectrogram_best.keras'
manifest_path = project_root / 'outputs' / 'noisy_drone_rf_v2_eval' / '33_noisy_drone_rf_v2_vgg_full_complex_replay_manifest.csv'
if not manifest_path.exists():
    manifest_path = project_root / 'outputs' / 'noisy_drone_rf_v2_eval' / '33_noisy_drone_rf_v2_replay_manifest.csv'

CLASS_NAMES = ['DJI', 'FutabaT14', 'FutabaT7', 'Graupner', 'Noise', 'Taranis', 'Turnigy']
NUM_CLASSES = len(CLASS_NAMES)

RANDOM_STATE = int(os.getenv('NOISY_DRONE_GAN_RANDOM_STATE', '42'))
DATA_FRACTION = float(os.getenv('NOISY_DRONE_GAN_DATA_FRACTION', '0.35'))
MIN_SNR_DB = float(os.getenv('NOISY_DRONE_GAN_MIN_SNR_DB', '-6'))
BATCH_SIZE = int(os.getenv('NOISY_DRONE_GAN_BATCH_SIZE', '16'))
EPOCHS = int(os.getenv('NOISY_DRONE_GAN_EPOCHS', '40'))
STEPS_PER_EPOCH = int(os.getenv('NOISY_DRONE_GAN_STEPS_PER_EPOCH', '300'))
LATENT_DIM = int(os.getenv('NOISY_DRONE_GAN_LATENT_DIM', '128'))
SIGNATURE_BINS = int(os.getenv('NOISY_DRONE_GAN_SIGNATURE_BINS', '128'))
SIGNATURE_TIME_BINS = int(os.getenv('NOISY_DRONE_GAN_SIGNATURE_TIME_BINS', '128'))
SAMPLE_LEN = int(os.getenv('NOISY_DRONE_GAN_SAMPLE_LEN', '262144'))
SPEC_FRAME_LEN = int(os.getenv('NOISY_DRONE_GAN_SPEC_FRAME_LEN', '512'))
GEN_LR = float(os.getenv('NOISY_DRONE_GAN_GEN_LR', '1e-4'))
DISC_LR = float(os.getenv('NOISY_DRONE_GAN_DISC_LR', '8e-5'))
TEACHER_WEIGHT = float(os.getenv('NOISY_DRONE_GAN_TEACHER_WEIGHT', '0.05'))
CLASS_WEIGHT = float(os.getenv('NOISY_DRONE_GAN_CLASS_WEIGHT', '1.00'))
NOISE_AVOID_WEIGHT = float(os.getenv('NOISY_DRONE_GAN_NOISE_AVOID_WEIGHT', '0.50'))
MOMENT_MATCH_WEIGHT = float(os.getenv('NOISY_DRONE_GAN_MOMENT_MATCH_WEIGHT', '1.00'))
SIGNATURE_L1_WEIGHT = float(os.getenv('NOISY_DRONE_GAN_SIGNATURE_L1_WEIGHT', '0.10'))
FREEZE_DISCRIMINATOR = os.getenv('NOISY_DRONE_GAN_FREEZE_DISCRIMINATOR', '0') == '1'
R1_WEIGHT = float(os.getenv('NOISY_DRONE_GAN_R1_WEIGHT', '0.0'))
SAVE_EVERY = int(os.getenv('NOISY_DRONE_GAN_SAVE_EVERY', '5'))

SIGNATURE_SHAPE = (SIGNATURE_BINS, SIGNATURE_TIME_BINS, 2)
TEACHER_SHAPE = (1024, 1024)

generator_path = model_dir / 'noisy_drone_rf_v2_conditional_signature_generator.keras'
discriminator_path = model_dir / 'noisy_drone_rf_v2_conditional_signature_discriminator.keras'
history_csv_path = outputs_dir / '63_noisy_drone_rf_v2_gan_training_history.csv'

print('Project root:', project_root)
print('Manifest:', manifest_path)
print('Teacher model:', teacher_model_path)
print('Signature shape:', SIGNATURE_SHAPE)
print('Batch:', BATCH_SIZE, 'epochs:', EPOCHS, 'steps/epoch:', STEPS_PER_EPOCH)
print('Teacher consistency weight:', TEACHER_WEIGHT)
print('Class/moment/L1/noise-avoid weights:', CLASS_WEIGHT, MOMENT_MATCH_WEIGHT, SIGNATURE_L1_WEIGHT, NOISE_AVOID_WEIGHT)

In [ ]:
# Cell 2: Load the 33 replay manifest and build a balanced training frame
if not manifest_path.exists():
    raise FileNotFoundError(
        f'Missing replay manifest: {manifest_path}. Run notebook 33 Cell 1 first, or point manifest_path at a compatible CSV.'
    )

manifest_df = pd.read_csv(manifest_path)
required = {'filepath', 'label_idx', 'snr'}
missing = required - set(manifest_df.columns)
if missing:
    raise ValueError(f'Manifest missing required columns: {missing}')

manifest_df = manifest_df.copy()
manifest_df['filepath'] = manifest_df['filepath'].astype(str)
manifest_df['label_idx'] = manifest_df['label_idx'].astype(int)
manifest_df['snr'] = manifest_df['snr'].astype(float)
manifest_df = manifest_df[manifest_df['snr'] >= MIN_SNR_DB]
manifest_df = manifest_df[manifest_df['label_idx'].between(0, NUM_CLASSES - 1)]
manifest_df = manifest_df[manifest_df['filepath'].map(lambda p: Path(p).exists())]
manifest_df['label'] = manifest_df['label_idx'].map(lambda i: CLASS_NAMES[int(i)])

if manifest_df.empty:
    raise FileNotFoundError('No usable Noisy Drone RF v2 rows after filtering; verify /scratch data and the manifest paths.')

if DATA_FRACTION < 1.0:
    manifest_df, _ = train_test_split(
        manifest_df,
        train_size=DATA_FRACTION,
        stratify=manifest_df['label_idx'],
        random_state=RANDOM_STATE,
    )

# Keep the GAN balanced. Synthetic minority class behavior is the entire point here.
min_count = int(manifest_df['label_idx'].value_counts().min())
balanced_df = (
    manifest_df.groupby('label_idx', group_keys=False)
    .sample(n=min_count, random_state=RANDOM_STATE)
    .sample(frac=1.0, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

train_df, val_df = train_test_split(
    balanced_df,
    test_size=0.15,
    stratify=balanced_df['label_idx'],
    random_state=RANDOM_STATE,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print('Filtered manifest rows:', len(manifest_df))
print('Balanced rows:', len(balanced_df), 'train:', len(train_df), 'val:', len(val_df))
print('Balanced class counts:')
print(balanced_df['label'].value_counts().sort_index())

balanced_manifest_path = outputs_dir / '63_noisy_drone_rf_v2_gan_balanced_manifest.csv'
balanced_df.to_csv(balanced_manifest_path, index=False)
print('Saved balanced manifest:', balanced_manifest_path)

In [ ]:
# Cell 3: Define robust I/Q loading and compact full-complex spectrogram preprocessing
def _unwrap_tensor_container(obj):
    # The drone dataset .pt files are not perfectly uniform; peel common dict/list wrappers.
    seen = set()
    while True:
        obj_id = id(obj)
        if obj_id in seen:
            break
        seen.add(obj_id)
        if isinstance(obj, dict):
            preferred = ['iq', 'x', 'data', 'samples', 'signal', 'arr', 'tensor']
            keys = list(obj.keys())
            picked = None
            for key in preferred:
                if key in obj:
                    picked = key
                    break
            if picked is None:
                tensorish = [k for k, v in obj.items() if hasattr(v, 'detach') or isinstance(v, (np.ndarray, list, tuple))]
                picked = tensorish[0] if tensorish else keys[0]
            obj = obj[picked]
            continue
        if isinstance(obj, (list, tuple)) and len(obj) == 1:
            obj = obj[0]
            continue
        break
    return obj

def load_pt_iq(filepath):
    obj = torch.load(filepath, map_location='cpu', weights_only=False)
    obj = _unwrap_tensor_container(obj)
    arr = obj.detach().cpu().numpy() if hasattr(obj, 'detach') else np.asarray(obj)

    if np.iscomplexobj(arr):
        arr = np.stack([arr.real, arr.imag], axis=-1)
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.squeeze(arr)

    if arr.ndim == 1:
        if arr.size % 2 != 0:
            arr = arr[:-1]
        arr = arr.reshape(-1, 2)
    elif arr.ndim == 2:
        if arr.shape[0] == 2 and arr.shape[1] != 2:
            arr = arr.T
        elif arr.shape[1] > 2:
            arr = arr[:, :2]
    elif arr.ndim >= 3:
        arr = arr.reshape(-1, arr.shape[-1])
        if arr.shape[1] > 2:
            arr = arr[:, :2]

    if arr.shape[1] != 2:
        raise ValueError(f'Could not coerce {filepath} into IQ shape (N, 2); got {arr.shape}')
    return arr.astype(np.float32)

def iq_to_signature_spectrogram(iq, snr_value=0.0):
    iq = np.asarray(iq, dtype=np.float32)
    if iq.shape[0] < SAMPLE_LEN:
        reps = int(np.ceil(SAMPLE_LEN / max(1, iq.shape[0])))
        iq = np.tile(iq, (reps, 1))[:SAMPLE_LEN]
    elif iq.shape[0] > SAMPLE_LEN:
        start = (iq.shape[0] - SAMPLE_LEN) // 2
        iq = iq[start:start + SAMPLE_LEN]

    complex_iq = iq[:, 0].astype(np.float32) + 1j * iq[:, 1].astype(np.float32)
    complex_iq = complex_iq - np.mean(complex_iq)
    scale = np.sqrt(np.mean(np.abs(complex_iq) ** 2)) + 1e-8
    complex_iq = complex_iq / scale

    max_start = max(0, len(complex_iq) - SPEC_FRAME_LEN)
    starts = np.linspace(0, max_start, SIGNATURE_TIME_BINS).round().astype(np.int64)
    window = np.hanning(SPEC_FRAME_LEN).astype(np.float32)
    frames = []
    for start in starts:
        frame = complex_iq[start:start + SPEC_FRAME_LEN]
        if len(frame) < SPEC_FRAME_LEN:
            frame = np.pad(frame, (0, SPEC_FRAME_LEN - len(frame)))
        spectrum = np.fft.fftshift(np.fft.fft(frame * window, n=SIGNATURE_BINS))
        frames.append(spectrum)
    spec = np.stack(frames, axis=1)
    spec = spec / (np.percentile(np.abs(spec), 95) + 1e-8)
    spec = np.clip(spec, -5.0, 5.0) / 5.0
    return np.stack([spec.real, spec.imag], axis=-1).astype(np.float32)

def row_to_signature(row):
    return iq_to_signature_spectrogram(load_pt_iq(row['filepath']), row.get('snr', 0.0))

preview = row_to_signature(train_df.iloc[0].to_dict())
print('Preview signature:', preview.shape, preview.dtype, float(preview.min()), float(preview.max()))

In [ ]:
# Cell 4: Build balanced tf.data streams for real signatures
def make_balanced_generator(frame, shuffle=True):
    frame = frame.copy().reset_index(drop=True)
    groups = {int(k): g.reset_index(drop=True) for k, g in frame.groupby('label_idx')}
    labels = sorted(groups)

    def gen():
        rng = np.random.default_rng()
        while True:
            label = int(rng.choice(labels))
            group = groups[label]
            row = group.iloc[int(rng.integers(0, len(group)))].to_dict()
            try:
                x = row_to_signature(row)
                yield x, np.int64(label)
            except Exception as exc:
                print(f'Skipping corrupt row: {row.get("filepath")} ({type(exc).__name__}: {exc})')
                continue
    return gen

def make_signature_dataset(frame, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_generator(
        make_balanced_generator(frame),
        output_signature=(
            tf.TensorSpec(shape=SIGNATURE_SHAPE, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_signature_dataset(train_df)
val_ds = make_signature_dataset(val_df)

x_batch, y_batch = next(iter(train_ds))
print('Batch:', x_batch.shape, y_batch.shape, 'labels:', y_batch.numpy()[:10].tolist())

In [ ]:
# Cell 5: Build the conditional generator, discriminator, and frozen 33 teacher
def build_generator(latent_dim=LATENT_DIM, num_classes=NUM_CLASSES):
    z_in = Input(shape=(latent_dim,), name='noise')
    y_in = Input(shape=(), dtype='int32', name='label')
    y = Embedding(num_classes, latent_dim, name='class_embedding')(y_in)
    y = Flatten(name='class_embedding_flat')(y)
    x = Concatenate(name='noise_plus_class')([z_in, y])
    x = Dense(8 * 8 * 256, use_bias=False, name='dense_seed')(x)
    x = BatchNormalization(name='seed_bn')(x)
    x = LeakyReLU(0.2, name='seed_lrelu')(x)
    x = Reshape((8, 8, 256), name='seed_reshape')(x)
    for filters, name in [(192, 'up1'), (128, 'up2'), (96, 'up3'), (64, 'up4')]:
        x = Conv2DTranspose(filters, 4, strides=2, padding='same', use_bias=False, name=f'{name}_deconv')(x)
        x = BatchNormalization(name=f'{name}_bn')(x)
        x = LeakyReLU(0.2, name=f'{name}_lrelu')(x)
    x = Conv2D(32, 3, padding='same', activation='relu', name='refine_conv')(x)
    out = Conv2D(2, 3, padding='same', activation='tanh', name='signature')(x)
    return Model([z_in, y_in], out, name='noisy_drone_signature_generator')

def build_discriminator(input_shape=SIGNATURE_SHAPE, num_classes=NUM_CLASSES):
    x_in = Input(shape=input_shape, name='signature_input')
    x = x_in
    for filters, name in [(64, 'down1'), (96, 'down2'), (128, 'down3'), (192, 'down4')]:
        x = Conv2D(filters, 4, strides=2, padding='same', name=f'{name}_conv')(x)
        x = LeakyReLU(0.2, name=f'{name}_lrelu')(x)
        x = Dropout(0.15, name=f'{name}_dropout')(x)
    x = Flatten(name='flat')(x)
    x = Dense(256, name='shared_dense')(x)
    x = LeakyReLU(0.2, name='shared_lrelu')(x)
    real_fake = Dense(1, activation='sigmoid', name='real_fake')(x)
    class_logits = Dense(num_classes, activation='softmax', name='class')(x)
    return Model(x_in, [real_fake, class_logits], name='noisy_drone_signature_discriminator')

generator = load_model(generator_path, compile=False) if generator_path.exists() else build_generator()
discriminator = load_model(discriminator_path, compile=False) if discriminator_path.exists() else build_discriminator()

if FREEZE_DISCRIMINATOR:
    discriminator.trainable = False
    print('Loaded discriminator in frozen continuation mode. It will score generator outputs but will not update.')
else:
    discriminator.trainable = True
    print('Loaded discriminator in adversarial training mode. It will continue updating.')

teacher_model = None
if TEACHER_WEIGHT > 0:
    if not teacher_model_path.exists():
        raise FileNotFoundError(f'Teacher requested but missing: {teacher_model_path}')
    teacher_model = load_model(teacher_model_path, compile=False)
    teacher_model.trainable = False
    print('Loaded frozen teacher:', teacher_model_path)
else:
    print('Teacher consistency disabled')

generator.summary()
discriminator.summary()

In [ ]:
# Cell 6: Train the teacher-guided conditional GAN
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
sce = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
g_optimizer = tf.keras.optimizers.Adam(learning_rate=GEN_LR, beta_1=0.5, beta_2=0.999)
d_optimizer = tf.keras.optimizers.Adam(learning_rate=DISC_LR, beta_1=0.5, beta_2=0.999)

@tf.function
def train_step(real_x, labels):
    batch_size = tf.shape(real_x)[0]
    noise = tf.random.normal([batch_size, LATENT_DIM])
    real_targets = tf.ones((batch_size, 1)) * 0.9
    fake_targets = tf.zeros((batch_size, 1))

    # In frozen-discriminator mode, the discriminator is only a fixed critic/class head.
    # This is useful when its class head is already strong but the frozen 33 VGG teacher
    # still sees generated signatures as Noise.
    if FREEZE_DISCRIMINATOR:
        fake_x_probe = generator([noise, labels], training=False)
        real_pred, real_cls = discriminator(real_x, training=False)
        fake_pred, fake_cls = discriminator(fake_x_probe, training=False)
        d_adv = bce(real_targets, real_pred) + bce(fake_targets, fake_pred)
        d_cls = sce(labels, real_cls)
        d_loss = d_adv + CLASS_WEIGHT * d_cls
    else:
        with tf.GradientTape() as disc_tape:
            fake_x_probe = generator([noise, labels], training=True)
            real_pred, real_cls = discriminator(real_x, training=True)
            fake_pred, fake_cls = discriminator(fake_x_probe, training=True)
            d_adv = bce(real_targets, real_pred) + bce(fake_targets, fake_pred)
            d_cls = sce(labels, real_cls)
            d_loss = d_adv + CLASS_WEIGHT * d_cls
        d_grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)
        d_optimizer.apply_gradients(zip(d_grads, discriminator.trainable_variables))

    noise = tf.random.normal([batch_size, LATENT_DIM])
    with tf.GradientTape() as gen_tape:
        fake_x = generator([noise, labels], training=True)
        fake_pred, fake_cls = discriminator(fake_x, training=False)
        g_adv = bce(tf.ones((batch_size, 1)), fake_pred)
        g_cls = sce(labels, fake_cls)

        # Anchor generated signatures to real batch statistics for the requested classes.
        # This is intentionally weaker than reconstruction: it prevents GAN drift/mode collapse
        # without forcing one generated sample to copy one arbitrary real sample.
        real_mean = tf.reduce_mean(real_x, axis=[1, 2, 3])
        fake_mean = tf.reduce_mean(fake_x, axis=[1, 2, 3])
        real_std = tf.math.reduce_std(real_x, axis=[1, 2, 3])
        fake_std = tf.math.reduce_std(fake_x, axis=[1, 2, 3])
        moment_match_loss = tf.reduce_mean(tf.abs(real_mean - fake_mean) + tf.abs(real_std - fake_std))
        signature_l1_loss = tf.reduce_mean(tf.abs(tf.stop_gradient(real_x) - fake_x))

        teacher_loss = tf.constant(0.0, dtype=tf.float32)
        noise_avoid_loss = tf.constant(0.0, dtype=tf.float32)
        if teacher_model is not None and TEACHER_WEIGHT > 0:
            teacher_x = tf.image.resize(fake_x, TEACHER_SHAPE, method='bilinear')
            teacher_probs = teacher_model(teacher_x, training=False)
            teacher_loss = sce(labels, teacher_probs)

            # The observed failure is teacher collapse to class 4 = Noise for every generated class.
            # Penalize high teacher Noise probability for requested non-noise signatures.
            non_noise_mask = tf.cast(tf.not_equal(labels, 4), tf.float32)
            p_noise = tf.clip_by_value(teacher_probs[:, 4], 1e-6, 1.0 - 1e-6)
            denom = tf.reduce_sum(non_noise_mask) + 1e-6
            noise_avoid_loss = tf.reduce_sum(-tf.math.log(1.0 - p_noise) * non_noise_mask) / denom
        g_loss = (
            g_adv
            + CLASS_WEIGHT * g_cls
            + TEACHER_WEIGHT * teacher_loss
            + NOISE_AVOID_WEIGHT * noise_avoid_loss
            + MOMENT_MATCH_WEIGHT * moment_match_loss
            + SIGNATURE_L1_WEIGHT * signature_l1_loss
        )

    g_grads = gen_tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(g_grads, generator.trainable_variables))

    return {
        'd_loss': d_loss,
        'd_adv': d_adv,
        'd_cls': d_cls,
        'g_loss': g_loss,
        'g_adv': g_adv,
        'g_cls': g_cls,
        'teacher_loss': teacher_loss,
        'noise_avoid_loss': noise_avoid_loss,
        'moment_match_loss': moment_match_loss,
        'signature_l1_loss': signature_l1_loss,
    }

def generate_fixed_grid(epoch, n_per_class=4):
    labels = np.repeat(np.arange(NUM_CLASSES), n_per_class).astype(np.int32)
    rng = np.random.default_rng(12345)
    noise = rng.standard_normal((len(labels), LATENT_DIM)).astype(np.float32)
    fake = generator.predict([noise, labels], batch_size=BATCH_SIZE, verbose=0)
    fig, axes = plt.subplots(NUM_CLASSES, n_per_class, figsize=(n_per_class * 2.5, NUM_CLASSES * 2.2))
    for row in range(NUM_CLASSES):
        for col in range(n_per_class):
            idx = row * n_per_class + col
            ax = axes[row, col]
            mag = np.sqrt(fake[idx, :, :, 0] ** 2 + fake[idx, :, :, 1] ** 2)
            ax.imshow(mag, aspect='auto', origin='lower', cmap='magma')
            ax.set_xticks([])
            ax.set_yticks([])
            if col == 0:
                ax.set_ylabel(CLASS_NAMES[row])
    fig.suptitle(f'Generated drone signatures - epoch {epoch}')
    fig.tight_layout()
    path = outputs_dir / f'63_generated_signature_grid_epoch_{epoch:03d}.png'
    fig.savefig(path, dpi=150)
    plt.show()
    plt.close(fig)
    return path

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
history_rows = []
iterator = iter(train_ds)

for epoch in range(1, EPOCHS + 1):
    epoch_metrics = []
    for step in range(1, STEPS_PER_EPOCH + 1):
        real_x, labels = next(iterator)
        metrics = train_step(real_x, labels)
        epoch_metrics.append({k: float(v.numpy()) for k, v in metrics.items()})
        if step == 1 or step % 50 == 0:
            print(
                f'epoch={epoch:03d} step={step:04d}/{STEPS_PER_EPOCH} '
                f"d_loss={epoch_metrics[-1]['d_loss']:.4f} g_loss={epoch_metrics[-1]['g_loss']:.4f} "
                f"teacher={epoch_metrics[-1]['teacher_loss']:.4f} "
                f"noise_avoid={epoch_metrics[-1].get('noise_avoid_loss', 0.0):.4f}"
            )

    row = {'run_id': run_id, 'epoch': epoch}
    for key in epoch_metrics[0]:
        row[key] = float(np.mean([m[key] for m in epoch_metrics]))
    history_rows.append(row)
    print('epoch summary:', row)

    if epoch == 1 or epoch % SAVE_EVERY == 0 or epoch == EPOCHS:
        generator.save(generator_path)
        discriminator.save(discriminator_path)
        grid_path = generate_fixed_grid(epoch)
        print('Saved generator:', generator_path)
        print('Saved discriminator:', discriminator_path)
        print('Saved generated grid:', grid_path)

history_df = pd.DataFrame(history_rows)
if history_csv_path.exists():
    history_df = pd.concat([pd.read_csv(history_csv_path), history_df], ignore_index=True)
history_df.to_csv(history_csv_path, index=False)
print('Saved GAN history:', history_csv_path)

In [ ]:
# Cell 7: Evaluate generated signatures with discriminator class head and frozen 33 teacher
if not generator_path.exists():
    raise FileNotFoundError(f'Missing generator checkpoint: {generator_path}')
if not discriminator_path.exists():
    raise FileNotFoundError(f'Missing discriminator checkpoint: {discriminator_path}')

generator = load_model(generator_path, compile=False)
discriminator = load_model(discriminator_path, compile=False)
if teacher_model_path.exists():
    teacher_model = load_model(teacher_model_path, compile=False)
    teacher_model.trainable = False
else:
    teacher_model = None

samples_per_class = int(os.getenv('NOISY_DRONE_GAN_EVAL_SAMPLES_PER_CLASS', '64'))
labels = np.repeat(np.arange(NUM_CLASSES), samples_per_class).astype(np.int32)
noise = np.random.default_rng(RANDOM_STATE).standard_normal((len(labels), LATENT_DIM)).astype(np.float32)
fake = generator.predict([noise, labels], batch_size=BATCH_SIZE, verbose=1)
real_fake, disc_class_probs = discriminator.predict(fake, batch_size=BATCH_SIZE, verbose=0)
disc_pred = disc_class_probs.argmax(axis=1)

eval_rows = []
for idx, class_name in enumerate(CLASS_NAMES):
    mask = labels == idx
    eval_rows.append({
        'label_idx': idx,
        'label': class_name,
        'disc_class_accuracy': float(np.mean(disc_pred[mask] == labels[mask])),
        'disc_realness_mean': float(np.mean(real_fake[mask])),
        'disc_realness_std': float(np.std(real_fake[mask])),
    })

if teacher_model is not None:
    teacher_x = tf.image.resize(fake, TEACHER_SHAPE, method='bilinear').numpy().astype(np.float32)
    teacher_probs = teacher_model.predict(teacher_x, batch_size=max(1, min(BATCH_SIZE, 8)), verbose=1)
    teacher_pred = teacher_probs.argmax(axis=1)
    for row in eval_rows:
        mask = labels == row['label_idx']
        row['teacher_class_accuracy'] = float(np.mean(teacher_pred[mask] == labels[mask]))
        row['teacher_target_confidence_mean'] = float(np.mean(teacher_probs[mask, row['label_idx']]))
else:
    teacher_pred = None

eval_df = pd.DataFrame(eval_rows)
eval_path = outputs_dir / '63_noisy_drone_rf_v2_gan_generated_signature_eval.csv'
eval_df.to_csv(eval_path, index=False)
print(eval_df)
print('Saved eval:', eval_path)

cm = pd.crosstab(pd.Series(labels, name='target'), pd.Series(disc_pred, name='disc_pred')).reindex(index=range(NUM_CLASSES), columns=range(NUM_CLASSES), fill_value=0)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_title('Generated Signatures - Discriminator Class Head')
ax.set_xlabel('Predicted generated class')
ax.set_ylabel('Requested generated class')
fig.tight_layout()
disc_cm_path = outputs_dir / '63_generated_signature_discriminator_confusion_matrix.png'
fig.savefig(disc_cm_path, dpi=150)
plt.show()
plt.close(fig)
print('Saved:', disc_cm_path)

if teacher_pred is not None:
    cm = pd.crosstab(pd.Series(labels, name='target'), pd.Series(teacher_pred, name='teacher_pred')).reindex(index=range(NUM_CLASSES), columns=range(NUM_CLASSES), fill_value=0)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title('Generated Signatures - Frozen 33 VGG Teacher')
    ax.set_xlabel('Teacher predicted class')
    ax.set_ylabel('Requested generated class')
    fig.tight_layout()
    teacher_cm_path = outputs_dir / '63_generated_signature_teacher_confusion_matrix.png'
    fig.savefig(teacher_cm_path, dpi=150)
    plt.show()
    plt.close(fig)
    print('Saved:', teacher_cm_path)

In [ ]:
# Cell 8: Export a compact synthetic signature bank for later augmentation experiments
if 'fake' not in globals() or 'labels' not in globals():
    generator = load_model(generator_path, compile=False)
    samples_per_class = int(os.getenv('NOISY_DRONE_GAN_EXPORT_SAMPLES_PER_CLASS', '256'))
    labels = np.repeat(np.arange(NUM_CLASSES), samples_per_class).astype(np.int32)
    noise = np.random.default_rng(RANDOM_STATE + 1).standard_normal((len(labels), LATENT_DIM)).astype(np.float32)
    fake = generator.predict([noise, labels], batch_size=BATCH_SIZE, verbose=1)

export_path = outputs_dir / '63_noisy_drone_rf_v2_generated_signature_bank.npz'
np.savez_compressed(
    export_path,
    signatures=fake.astype(np.float32),
    labels=labels.astype(np.int64),
    class_names=np.asarray(CLASS_NAMES),
    signature_shape=np.asarray(SIGNATURE_SHAPE),
)
metadata = {
    'generator_path': str(generator_path),
    'teacher_model_path': str(teacher_model_path),
    'signature_shape': list(SIGNATURE_SHAPE),
    'latent_dim': LATENT_DIM,
    'samples': int(len(labels)),
    'class_names': CLASS_NAMES,
    'notes': 'Synthetic compact full-complex spectrogram signatures; not raw IQ.',
}
metadata_path = outputs_dir / '63_noisy_drone_rf_v2_generated_signature_bank.json'
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved signature bank:', export_path)
print('Saved metadata:', metadata_path)

In [ ]:
# Cell 9: Plot GAN training curves
if not history_csv_path.exists():
    raise FileNotFoundError(f'Missing GAN history: {history_csv_path}')

history_df = pd.read_csv(history_csv_path).copy()
history_df['global_epoch'] = np.arange(1, len(history_df) + 1)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for col in ['d_loss', 'g_loss']:
    if col in history_df:
        axes[0].plot(history_df['global_epoch'], history_df[col], marker='o', label=col)
axes[0].set_title('Noisy Drone RF v2 Conditional GAN Loss')
axes[0].set_xlabel('Global epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.35)
axes[0].legend()

for col in ['d_adv', 'd_cls', 'g_adv', 'g_cls', 'teacher_loss']:
    if col in history_df:
        axes[1].plot(history_df['global_epoch'], history_df[col], marker='o', label=col)
axes[1].set_title('Noisy Drone RF v2 Conditional GAN Components')
axes[1].set_xlabel('Global epoch')
axes[1].set_ylabel('Loss component')
axes[1].grid(True, alpha=0.35)
axes[1].legend()
fig.tight_layout()
plot_path = outputs_dir / '63_noisy_drone_rf_v2_gan_training_curves.png'
fig.savefig(plot_path, dpi=150)
plt.show()
plt.close(fig)
print('Saved:', plot_path)